In [1]:
#!/usr/bin/env python3
"""
Diagnose Fisher information issues for logistic regression on neural animate/inanimate data.

Main question
-------------
Does logistic regression have Fisher/Hessian pathologies on this data?

Data regime
-----------
n_stimuli = 118
n_neurons ~ 39209

For logistic regression:

    p_i = sigmoid(x_i @ w + b)
    W_ii = p_i * (1 - p_i)

Observed Fisher / Hessian wrt weights:

    I = X.T @ W @ X

Because p >> n, unregularized I is rank-deficient:

    rank(I) <= n

This script computes the nonzero spectrum using the dual matrix:

    G = sqrt(W) X X.T sqrt(W)

G is n x n and has the same nonzero eigenvalues as X.T W X.

Outputs
-------
/home/maria/Science/thesis/experiments/007--PoorMansClassifier/
    logistic_fisher_diagnostics/
        fisher_diagnostics_summary.json
        logistic_predictions.csv
        fisher_spectrum.csv
        fisher_spectrum.png
        fisher_spectrum_log.png
        probability_histogram.png
        poor_vs_logistic_weights.csv
"""

from __future__ import annotations

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
)
from sklearn.model_selection import LeaveOneOut, train_test_split
from sklearn.preprocessing import StandardScaler


# =============================================================================
# Config
# =============================================================================

BASE_DIR = Path("/home/maria/Science/thesis/experiments/007--PoorMansClassifier")
DATA_DIR = Path("/home/maria/Science/data")

OUT_DIR = BASE_DIR / "logistic_fisher_diagnostics"
OUT_DIR.mkdir(exist_ok=True, parents=True)

NEURAL_FILE = DATA_DIR / "hybrid_neural_responses_reduced.npy"
VIT_FILE = DATA_DIR / "google_vit-base-patch16-224_embeddings_logits.pkl"
VIT_KEY = "natural_scenes"

N_STIMULI = 118
ANIMATE_TOP1_THRESHOLD = 397

PRESENTATION_ORDER = "block"
STIMULUS_IDS_FILE = DATA_DIR / "stimulus_ids.npy"

RANDOM_SEED = 42
TEST_SIZE = 0.2

EPS = 1e-12

# Preprocessing choices.
# This matches your normalized poor man's classifier.
ROW_L2_NORMALIZE = True

# Strong recommendation for logistic regression:
# z-scoring makes logistic weights more interpretable and optimization better-conditioned.
STANDARDIZE_NEURONS_FOR_LOGISTIC = True

# Logistic regression regularization.
# Smaller C = stronger L2 regularization.
LOGISTIC_C = 1.0
LOGISTIC_MAX_ITER = 5000

# For sklearn:
# lbfgs is good for dense small-n / high-p if memory is okay.
# liblinear can also work for binary classification.
LOGISTIC_SOLVER = "lbfgs"

RUN_LOO_LOGISTIC = True


# =============================================================================
# Loading
# =============================================================================

def load_neural_presentations() -> np.ndarray:
    if not NEURAL_FILE.exists():
        raise FileNotFoundError(f"Missing neural file: {NEURAL_FILE}")

    X_raw = np.asarray(np.load(NEURAL_FILE, allow_pickle=True))
    print(f"[INFO] Raw neural shape: {X_raw.shape}")

    if X_raw.ndim != 2:
        raise ValueError(f"Expected 2D neural matrix, got {X_raw.shape}")

    n0, n1 = X_raw.shape

    if n0 > n1 and n1 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as neurons x presentations.")
        X_pres = X_raw.T
    elif n1 > n0 and n0 % N_STIMULI == 0:
        print("[INFO] Interpreting raw neural matrix as presentations x neurons.")
        X_pres = X_raw
    else:
        raise ValueError(
            f"Could not infer orientation from neural shape {X_raw.shape}. "
            "Expected something like (39209, 118) or (118, 39209)."
        )

    X_pres = X_pres.astype(np.float32, copy=False)
    print(f"[INFO] Presentation-level neural shape: {X_pres.shape}")
    return X_pres


def load_vit_natural_scenes_logits() -> np.ndarray:
    if not VIT_FILE.exists():
        raise FileNotFoundError(f"Missing ViT file: {VIT_FILE}")

    obj = np.load(VIT_FILE, allow_pickle=True)

    if hasattr(obj, "keys"):
        if VIT_KEY not in obj.keys():
            raise KeyError(f"Key {VIT_KEY!r} not found in {VIT_FILE}")
        logits = np.asarray(obj[VIT_KEY])
    elif isinstance(obj, np.ndarray) and obj.dtype == object:
        item = obj.item()
        if not isinstance(item, dict):
            raise TypeError(f"Expected object array containing dict, got {type(item)}")
        if VIT_KEY not in item:
            raise KeyError(f"Key {VIT_KEY!r} not found in object dict.")
        logits = np.asarray(item[VIT_KEY])
    else:
        raise TypeError(f"Unsupported ViT object type: {type(obj)}")

    if logits.ndim != 2:
        raise ValueError(f"Expected 2D ViT logits, got {logits.shape}")

    if logits.shape[0] != N_STIMULI:
        raise ValueError(f"Expected {N_STIMULI} rows, got {logits.shape[0]}")

    print(f"[INFO] ViT logits shape: {logits.shape}")
    return logits.astype(np.float32, copy=False)


def make_labels_from_vit_logits(logits: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    top1 = np.argmax(logits, axis=1)
    y = (top1 <= ANIMATE_TOP1_THRESHOLD).astype(int)

    print("[INFO] Derived animate/inanimate labels from ViT top-1.")
    print(f"[INFO] Inanimate count: {int((y == 0).sum())}")
    print(f"[INFO] Animate count:   {int((y == 1).sum())}")

    return y, top1


# =============================================================================
# Stimulus averaging
# =============================================================================

def make_presentation_stimulus_ids(n_presentations: int) -> np.ndarray:
    if STIMULUS_IDS_FILE.exists():
        stim_ids = np.load(STIMULUS_IDS_FILE, allow_pickle=True).astype(int).ravel()

        if len(stim_ids) != n_presentations:
            raise ValueError(
                f"{STIMULUS_IDS_FILE} has length {len(stim_ids)}, "
                f"but neural data has {n_presentations} presentations."
            )

        if stim_ids.min() < 0 or stim_ids.max() >= N_STIMULI:
            raise ValueError(
                f"Stimulus IDs must be in [0, {N_STIMULI - 1}], "
                f"got min={stim_ids.min()}, max={stim_ids.max()}."
            )

        print(f"[INFO] Loaded explicit stimulus IDs from {STIMULUS_IDS_FILE}")
        return stim_ids

    if n_presentations % N_STIMULI != 0:
        raise ValueError(
            f"n_presentations={n_presentations} is not divisible by {N_STIMULI}."
        )

    repeats = n_presentations // N_STIMULI

    if PRESENTATION_ORDER == "block":
        stim_ids = np.repeat(np.arange(N_STIMULI), repeats)
    elif PRESENTATION_ORDER == "cycle":
        stim_ids = np.tile(np.arange(N_STIMULI), repeats)
    else:
        raise ValueError("PRESENTATION_ORDER must be either 'block' or 'cycle'.")

    print(
        f"[WARN] No explicit {STIMULUS_IDS_FILE.name} found. "
        f"Assuming PRESENTATION_ORDER={PRESENTATION_ORDER!r}. "
        f"Repeats per stimulus={repeats}."
    )

    return stim_ids.astype(int)


def average_presentations_by_stimulus(X_pres: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    n_presentations, n_neurons = X_pres.shape
    stim_ids = make_presentation_stimulus_ids(n_presentations)

    X_avg = np.zeros((N_STIMULI, n_neurons), dtype=np.float32)
    counts = np.zeros(N_STIMULI, dtype=int)

    for stim_id in range(N_STIMULI):
        mask = stim_ids == stim_id
        counts[stim_id] = int(mask.sum())

        if counts[stim_id] == 0:
            raise ValueError(f"Stimulus {stim_id} has zero presentations.")

        X_avg[stim_id] = X_pres[mask].mean(axis=0)

    print("[INFO] Averaged neural responses by stimulus.")
    print(f"[INFO] Stimulus-averaged neural shape: {X_avg.shape}")
    print(f"[INFO] Presentations per stimulus: min={counts.min()}, max={counts.max()}")

    return X_avg, counts


# =============================================================================
# Preprocessing
# =============================================================================

def clean_features_all_data(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    finite = np.isfinite(X).all(axis=0)
    nonzero_var = np.nanvar(X, axis=0) > 0

    keep = finite & nonzero_var
    kept_original_indices = np.where(keep)[0]

    removed = X.shape[1] - int(keep.sum())
    if removed:
        print(f"[WARN] Removing {removed} non-finite or zero-variance neurons.")

    X_clean = X[:, keep].astype(np.float32, copy=False)

    print(f"[INFO] Clean stimulus-level neural shape: {X_clean.shape}")

    pd.DataFrame(
        {
            "clean_feature_index": np.arange(len(kept_original_indices)),
            "original_neuron_index": kept_original_indices,
        }
    ).to_csv(OUT_DIR / "kept_neuron_indices.csv", index=False)

    return X_clean, kept_original_indices


def l2_normalize_rows(X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms_safe = np.maximum(norms, EPS)
    X_hat = X / norms_safe
    return X_hat.astype(np.float32, copy=False), norms.ravel().astype(np.float32)


def preprocess_for_logistic_train_test(
    X_train: np.ndarray,
    X_test: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Apply optional row normalization and train-fitted standardization.
    """
    info = {}

    if ROW_L2_NORMALIZE:
        X_train, train_norms = l2_normalize_rows(X_train)
        X_test, test_norms = l2_normalize_rows(X_test)

        info["row_l2_normalize"] = True
        info["train_row_norms_before_normalization"] = {
            "min": float(train_norms.min()),
            "median": float(np.median(train_norms)),
            "max": float(train_norms.max()),
        }
        info["test_row_norms_before_normalization"] = {
            "min": float(test_norms.min()),
            "median": float(np.median(test_norms)),
            "max": float(test_norms.max()),
        }
    else:
        info["row_l2_normalize"] = False

    if STANDARDIZE_NEURONS_FOR_LOGISTIC:
        scaler = StandardScaler(with_mean=True, with_std=True)
        X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)
        X_test = scaler.transform(X_test).astype(np.float32, copy=False)

        info["standardize_neurons_for_logistic"] = True
    else:
        scaler = None
        info["standardize_neurons_for_logistic"] = False

    info["scaler_used"] = scaler is not None

    return X_train, X_test, info


# =============================================================================
# Metrics
# =============================================================================

def safe_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, scores))


def metric_dict(y_true: np.ndarray, scores: np.ndarray, preds: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, preds, labels=[0, 1])

    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, preds)),
        "auc": safe_auc(y_true, scores),
        "confusion_matrix_labels": ["inanimate_0", "animate_1"],
        "confusion_matrix": cm.tolist(),
    }


# =============================================================================
# Poor man's normalized direction for comparison
# =============================================================================

def normalized_poor_mans_direction(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    X should already be preprocessed like logistic input.
    Computes class mean directions and returns mu1_hat - mu0_hat.
    """
    mu0 = X[y == 0].mean(axis=0)
    mu1 = X[y == 1].mean(axis=0)

    mu0_norm = max(float(np.linalg.norm(mu0)), EPS)
    mu1_norm = max(float(np.linalg.norm(mu1)), EPS)

    mu0_hat = mu0 / mu0_norm
    mu1_hat = mu1 / mu1_norm

    return (mu1_hat - mu0_hat).astype(np.float32, copy=False)


# =============================================================================
# Logistic and Fisher diagnostics
# =============================================================================

def fit_logistic_regression(
    X_train: np.ndarray,
    y_train: np.ndarray,
) -> tuple[LogisticRegression, bool, str | None]:
    """
    Fit L2-regularized logistic regression and catch convergence warnings.
    """
    clf = LogisticRegression(
        penalty="l2",
        C=LOGISTIC_C,
        solver=LOGISTIC_SOLVER,
        max_iter=LOGISTIC_MAX_ITER,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        fit_intercept=True,
    )

    converged = True
    warning_message = None

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        clf.fit(X_train, y_train)

        for w in caught:
            if issubclass(w.category, ConvergenceWarning):
                converged = False
                warning_message = str(w.message)

    return clf, converged, warning_message


def fisher_spectrum_dual(
    X: np.ndarray,
    probs: np.ndarray,
    include_intercept: bool = False,
) -> dict:
    """
    Compute nonzero eigenvalues of X.T @ W @ X using n x n dual matrix.

    W_i = p_i * (1 - p_i)

    If include_intercept=True, append a column of ones to X.
    """
    n, p = X.shape

    if include_intercept:
        X_use = np.concatenate(
            [X, np.ones((n, 1), dtype=X.dtype)],
            axis=1,
        )
    else:
        X_use = X

    p_effective = X_use.shape[1]

    fisher_weights = probs * (1.0 - probs)

    sqrt_w = np.sqrt(np.maximum(fisher_weights, 0.0)).astype(np.float64)

    Xw = X_use.astype(np.float64, copy=False) * sqrt_w[:, None]

    # Nonzero eigenvalues of X.T W X are eigenvalues of Xw Xw.T.
    G = Xw @ Xw.T
    G = 0.5 * (G + G.T)

    evals = np.linalg.eigvalsh(G)
    evals = np.sort(evals)[::-1]

    tol = max(G.shape) * np.finfo(float).eps * max(float(evals[0]), 1.0)
    positive = evals[evals > tol]

    rank = int(len(positive))
    null_dim = int(p_effective - rank)

    if rank >= 2:
        condition_nonzero = float(positive[0] / positive[-1])
    else:
        condition_nonzero = float("inf")

    diagnostics = {
        "n_samples": int(n),
        "n_features_effective": int(p_effective),
        "rank_estimate": rank,
        "nullspace_dimension_estimate": null_dim,
        "tol": float(tol),
        "max_eigenvalue": float(evals[0]) if len(evals) else 0.0,
        "min_positive_eigenvalue": float(positive[-1]) if len(positive) else 0.0,
        "condition_number_nonzero_spectrum": condition_nonzero,
        "n_eigenvalues_returned": int(len(evals)),
        "fisher_weight_min": float(fisher_weights.min()),
        "fisher_weight_median": float(np.median(fisher_weights)),
        "fisher_weight_max": float(fisher_weights.max()),
        "fisher_weight_mean": float(fisher_weights.mean()),
        "n_saturated_prob_lt_001": int(np.sum(probs < 0.01)),
        "n_saturated_prob_gt_099": int(np.sum(probs > 0.99)),
        "n_low_fisher_weight_lt_1e_minus_4": int(np.sum(fisher_weights < 1e-4)),
    }

    return {
        "eigenvalues": evals,
        "positive_eigenvalues": positive,
        "diagnostics": diagnostics,
        "fisher_weights": fisher_weights,
    }


def regularized_fisher_diagnostics(
    unregularized_positive_evals: np.ndarray,
    n_features: int,
    rank: int,
    alpha: float,
) -> dict:
    """
    For ridge/L2-regularized Hessian:

        H_reg = X.T W X + alpha I

    All nullspace eigenvalues become alpha.
    Existing positive eigenvalues become eval + alpha.

    This reports condition of regularized Hessian.
    """
    if len(unregularized_positive_evals) > 0:
        max_eval_reg = float(unregularized_positive_evals[0] + alpha)
    else:
        max_eval_reg = float(alpha)

    min_eval_reg = float(alpha)

    return {
        "alpha_used": float(alpha),
        "max_regularized_eigenvalue": max_eval_reg,
        "min_regularized_eigenvalue": min_eval_reg,
        "regularized_condition_number": float(max_eval_reg / min_eval_reg),
        "nullspace_eigenvalues_equal_alpha": int(n_features - rank),
    }


def save_spectrum_plots(evals: np.ndarray) -> None:
    df = pd.DataFrame(
        {
            "index": np.arange(1, len(evals) + 1),
            "eigenvalue": evals,
        }
    )
    df.to_csv(OUT_DIR / "fisher_spectrum.csv", index=False)

    plt.figure(figsize=(8, 5))
    plt.plot(np.arange(1, len(evals) + 1), evals, marker="o", linewidth=1)
    plt.title("Observed logistic Fisher spectrum via dual matrix")
    plt.xlabel("Eigenvalue index")
    plt.ylabel("Eigenvalue")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fisher_spectrum.png", dpi=200)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.semilogy(np.arange(1, len(evals) + 1), np.maximum(evals, EPS), marker="o", linewidth=1)
    plt.title("Observed logistic Fisher spectrum, log scale")
    plt.xlabel("Eigenvalue index")
    plt.ylabel("Eigenvalue, log scale")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "fisher_spectrum_log.png", dpi=200)
    plt.close()


def save_probability_histogram(probs: np.ndarray) -> None:
    plt.figure(figsize=(8, 5))
    plt.hist(probs, bins=30)
    plt.axvline(0.5, linestyle="--", linewidth=1)
    plt.title("Logistic regression predicted probabilities")
    plt.xlabel("Predicted probability of animate")
    plt.ylabel("Stimulus count")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "probability_histogram.png", dpi=200)
    plt.close()


# =============================================================================
# LOO logistic
# =============================================================================

def run_loo_logistic(X: np.ndarray, y: np.ndarray) -> dict:
    loo = LeaveOneOut()

    scores = np.zeros(len(y), dtype=np.float32)
    preds = np.zeros(len(y), dtype=int)
    convergence_flags = []

    for fold_idx, (train_idx, test_idx) in enumerate(loo.split(X), start=1):
        X_train_raw = X[train_idx]
        X_test_raw = X[test_idx]
        y_train = y[train_idx]

        X_train, X_test, _ = preprocess_for_logistic_train_test(X_train_raw, X_test_raw)

        clf, converged, warning_message = fit_logistic_regression(X_train, y_train)
        convergence_flags.append(converged)

        prob = clf.predict_proba(X_test)[0, 1]
        scores[test_idx[0]] = prob
        preds[test_idx[0]] = int(prob >= 0.5)

        if fold_idx == 1 or fold_idx % 25 == 0 or fold_idx == len(y):
            print(
                f"[LOO] {fold_idx:3d}/{len(y)} "
                f"held_out={test_idx[0]:3d} "
                f"prob={prob:.4f} "
                f"pred={preds[test_idx[0]]} "
                f"true={y[test_idx[0]]} "
                f"converged={converged}"
            )
            if warning_message:
                print(f"      warning: {warning_message}")

    metrics = metric_dict(y, scores, preds)

    return {
        "scores": scores,
        "predictions": preds,
        "metrics": metrics,
        "n_converged_folds": int(np.sum(convergence_flags)),
        "n_folds": int(len(convergence_flags)),
    }


# =============================================================================
# Main
# =============================================================================

def main() -> None:
    print("=" * 80)
    print("Loading data")
    print("=" * 80)

    X_pres = load_neural_presentations()
    X_avg, presentation_counts = average_presentations_by_stimulus(X_pres)

    logits = load_vit_natural_scenes_logits()
    y, top1 = make_labels_from_vit_logits(logits)

    if X_avg.shape[0] != len(y):
        raise ValueError(f"X has {X_avg.shape[0]} rows, y has {len(y)} labels.")

    print("=" * 80)
    print("Cleaning features")
    print("=" * 80)

    X_clean, kept_original_indices = clean_features_all_data(X_avg)

    print("=" * 80)
    print("Train/test split")
    print("=" * 80)

    indices = np.arange(len(y))

    train_idx, test_idx = train_test_split(
        indices,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED,
        shuffle=True,
    )

    X_train_raw = X_clean[train_idx]
    X_test_raw = X_clean[test_idx]
    y_train = y[train_idx]
    y_test = y[test_idx]

    print(
        f"[INFO] Train n={len(train_idx)}, test n={len(test_idx)}, "
        f"features={X_clean.shape[1]}"
    )

    print("=" * 80)
    print("Preprocessing for logistic")
    print("=" * 80)

    X_train, X_test, preprocess_info = preprocess_for_logistic_train_test(
        X_train_raw,
        X_test_raw,
    )

    print(json.dumps(preprocess_info, indent=2))

    print("=" * 80)
    print("Fit logistic regression")
    print("=" * 80)

    clf, converged, warning_message = fit_logistic_regression(X_train, y_train)

    train_probs = clf.predict_proba(X_train)[:, 1]
    test_probs = clf.predict_proba(X_test)[:, 1]

    train_preds = (train_probs >= 0.5).astype(int)
    test_preds = (test_probs >= 0.5).astype(int)

    train_metrics = metric_dict(y_train, train_probs, train_preds)
    test_metrics = metric_dict(y_test, test_probs, test_preds)

    print("[RESULT] Train metrics:")
    print(json.dumps(train_metrics, indent=2))

    print("[RESULT] Test metrics:")
    print(json.dumps(test_metrics, indent=2))

    print(f"[INFO] Converged: {converged}")
    if warning_message:
        print(f"[WARN] {warning_message}")

    print("=" * 80)
    print("Fisher diagnostics on train set")
    print("=" * 80)

    fisher = fisher_spectrum_dual(
        X=X_train,
        probs=train_probs,
        include_intercept=False,
    )

    evals = fisher["eigenvalues"]
    positive = fisher["positive_eigenvalues"]
    fisher_diag = fisher["diagnostics"]

    # Approximate regularization alpha.
    # sklearn's exact objective scaling differs by solver conventions,
    # but alpha = 1/C is a useful Hessian-scale diagnostic.
    alpha = 1.0 / LOGISTIC_C

    reg_diag = regularized_fisher_diagnostics(
        unregularized_positive_evals=positive,
        n_features=X_train.shape[1],
        rank=fisher_diag["rank_estimate"],
        alpha=alpha,
    )

    print("[FISHER] Unregularized observed Fisher:")
    print(json.dumps(fisher_diag, indent=2))

    print("[FISHER] L2-regularized Hessian approximation:")
    print(json.dumps(reg_diag, indent=2))

    save_spectrum_plots(evals)
    save_probability_histogram(train_probs)

    print("=" * 80)
    print("Compare poor man's direction and logistic direction")
    print("=" * 80)

    w_logistic = clf.coef_.ravel().astype(np.float32)
    w_poor = normalized_poor_mans_direction(X_train, y_train)

    dot = float(np.dot(w_logistic, w_poor))
    norm_log = float(np.linalg.norm(w_logistic))
    norm_poor = float(np.linalg.norm(w_poor))
    cosine = dot / max(norm_log * norm_poor, EPS)

    print(f"[INFO] cosine(logistic w, poor man's w) = {cosine:.6f}")

    weights_df = pd.DataFrame(
        {
            "clean_feature_index": np.arange(X_train.shape[1]),
            "original_neuron_index": kept_original_indices,
            "logistic_weight": w_logistic,
            "poor_mans_weight": w_poor,
            "abs_logistic_weight": np.abs(w_logistic),
            "abs_poor_mans_weight": np.abs(w_poor),
        }
    )
    weights_df.to_csv(OUT_DIR / "poor_vs_logistic_weights.csv", index=False)

    print("=" * 80)
    print("Save train/test predictions")
    print("=" * 80)

    pred_rows = []

    for local_i, stim_idx in enumerate(train_idx):
        pred_rows.append(
            {
                "stimulus_index": int(stim_idx),
                "split": "train",
                "label_animate": int(y[stim_idx]),
                "top1_imagenet_class": int(top1[stim_idx]),
                "logistic_prob_animate": float(train_probs[local_i]),
                "logistic_prediction_animate": int(train_preds[local_i]),
                "correct": bool(train_preds[local_i] == y[stim_idx]),
            }
        )

    for local_i, stim_idx in enumerate(test_idx):
        pred_rows.append(
            {
                "stimulus_index": int(stim_idx),
                "split": "test",
                "label_animate": int(y[stim_idx]),
                "top1_imagenet_class": int(top1[stim_idx]),
                "logistic_prob_animate": float(test_probs[local_i]),
                "logistic_prediction_animate": int(test_preds[local_i]),
                "correct": bool(test_preds[local_i] == y[stim_idx]),
            }
        )

    pd.DataFrame(pred_rows).sort_values("stimulus_index").to_csv(
        OUT_DIR / "logistic_predictions.csv",
        index=False,
    )

    loo_summary = None

    if RUN_LOO_LOGISTIC:
        print("=" * 80)
        print("Running LOO logistic regression")
        print("=" * 80)

        loo_result = run_loo_logistic(X_clean, y)
        loo_summary = {
            "metrics": loo_result["metrics"],
            "n_converged_folds": loo_result["n_converged_folds"],
            "n_folds": loo_result["n_folds"],
        }

        pd.DataFrame(
            {
                "stimulus_index": np.arange(len(y)),
                "label_animate": y,
                "top1_imagenet_class": top1,
                "loo_logistic_prob_animate": loo_result["scores"],
                "loo_logistic_prediction_animate": loo_result["predictions"],
                "correct": loo_result["predictions"] == y,
            }
        ).to_csv(OUT_DIR / "loo_logistic_predictions.csv", index=False)

        print("[RESULT] LOO logistic metrics:")
        print(json.dumps(loo_summary, indent=2))

    summary = {
        "experiment": "logistic_fisher_diagnostics",
        "neural_file": str(NEURAL_FILE),
        "vit_file": str(VIT_FILE),
        "n_stimuli": int(N_STIMULI),
        "presentation_level_shape": list(X_pres.shape),
        "stimulus_averaged_shape": list(X_avg.shape),
        "clean_shape": list(X_clean.shape),
        "n_clean_neurons": int(X_clean.shape[1]),
        "class_counts": {
            "inanimate": int((y == 0).sum()),
            "animate": int((y == 1).sum()),
        },
        "presentation_order_assumption": PRESENTATION_ORDER,
        "used_explicit_stimulus_ids": bool(STIMULUS_IDS_FILE.exists()),
        "min_presentations_per_stimulus": int(presentation_counts.min()),
        "max_presentations_per_stimulus": int(presentation_counts.max()),
        "test_size": float(TEST_SIZE),
        "logistic": {
            "solver": LOGISTIC_SOLVER,
            "C": float(LOGISTIC_C),
            "max_iter": int(LOGISTIC_MAX_ITER),
            "class_weight": "balanced",
            "converged": bool(converged),
            "warning_message": warning_message,
        },
        "preprocessing": preprocess_info,
        "train_metrics": train_metrics,
        "test_metrics": test_metrics,
        "fisher_unregularized_train": fisher_diag,
        "fisher_l2_regularized_approximation": reg_diag,
        "logistic_weight_norm": norm_log,
        "poor_mans_weight_norm": norm_poor,
        "cosine_logistic_vs_poor_mans_direction": float(cosine),
        "loo_logistic": loo_summary,
        "interpretation_flags": {
            "p_greater_than_n": bool(X_train.shape[1] > X_train.shape[0]),
            "unregularized_fisher_singular_by_dimension": bool(X_train.shape[1] > fisher_diag["rank_estimate"]),
            "many_saturated_probabilities": bool(
                fisher_diag["n_saturated_prob_lt_001"] + fisher_diag["n_saturated_prob_gt_099"] > 0
            ),
            "very_low_fisher_weights_present": bool(
                fisher_diag["n_low_fisher_weight_lt_1e_minus_4"] > 0
            ),
        },
        "output_files": {
            "summary": str(OUT_DIR / "fisher_diagnostics_summary.json"),
            "predictions": str(OUT_DIR / "logistic_predictions.csv"),
            "loo_predictions": str(OUT_DIR / "loo_logistic_predictions.csv"),
            "fisher_spectrum": str(OUT_DIR / "fisher_spectrum.csv"),
            "fisher_spectrum_png": str(OUT_DIR / "fisher_spectrum.png"),
            "fisher_spectrum_log_png": str(OUT_DIR / "fisher_spectrum_log.png"),
            "probability_histogram_png": str(OUT_DIR / "probability_histogram.png"),
            "poor_vs_logistic_weights": str(OUT_DIR / "poor_vs_logistic_weights.csv"),
        },
    }

    with open(OUT_DIR / "fisher_diagnostics_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("=" * 80)
    print("Final summary")
    print("=" * 80)
    print(json.dumps(summary, indent=2))

    print("=" * 80)
    print(f"Done. Results saved to: {OUT_DIR}")
    print("=" * 80)


if __name__ == "__main__":
    main()

Loading data
[INFO] Raw neural shape: (39209, 118)
[INFO] Interpreting raw neural matrix as neurons x presentations.
[INFO] Presentation-level neural shape: (118, 39209)
[WARN] No explicit stimulus_ids.npy found. Assuming PRESENTATION_ORDER='block'. Repeats per stimulus=1.
[INFO] Averaged neural responses by stimulus.
[INFO] Stimulus-averaged neural shape: (118, 39209)
[INFO] Presentations per stimulus: min=1, max=1
[INFO] ViT logits shape: (118, 1000)
[INFO] Derived animate/inanimate labels from ViT top-1.
[INFO] Inanimate count: 55
[INFO] Animate count:   63
Cleaning features
[INFO] Clean stimulus-level neural shape: (118, 39209)
Train/test split
[INFO] Train n=94, test n=24, features=39209
Preprocessing for logistic
{
  "row_l2_normalize": true,
  "train_row_norms_before_normalization": {
    "min": 11.017495155334473,
    "median": 15.571321487426758,
    "max": 21.99540901184082
  },
  "test_row_norms_before_normalization": {
    "min": 13.131230354309082,
    "median": 15.5479850